In [24]:
import torch 
import torch.nn as nn

class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, bias=False):
        super().__init__()

        self.wq = nn.Linear(d_in, d_out, bias)
        self.wk = nn.Linear(d_in, d_out, bias)
        self.wv = nn.Linear(d_in, d_out, bias)

        self.drop_out = nn.Dropout(dropout)
        print(f"In init, context_length: {context_length}")

        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )
    def forward(self, x):
        batch_size, num_tokens, d_in=x.shape
        # Tensor의 크기를 본다, batch 사이크 token의 개수, token의 Dimension

        #####################################
        #### 예상 문제 1. [Q, K, V]를 계산한다.
        #####################################
        queries = self.wq(x)
        keys = self.wk(x)
        values = self.wv(x)

        print(f"In forward, context_length: {context_length}")
        print(f"In forward, num_tokens: {num_tokens}")

        ######################################
        ##### 예상 문제 2. Queries와 Keys를 내적해서 토큰 간의 관련성을 구한다. (MatMul)
        #######################################
        # Query와 Key를 내적해서 토큰 간의 관련성을 구한다.
        atten_scores = queries @ keys.transpose(1,2)

        # Mask가 1인 위치 중에서 매래 시점(오른쪽 대각선 위)을 -무한대로 채운다.
        ######################################
        ##### 예상 문제 3. 미래시점의을 -무한대로 채운다.
        ##### 무한대로 채우는 이유는 Softmax를 하면 0이 되버리기 때문임. 
        #######################################
        #

        #mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        #atten_scores.masked_fill(mask_bool, -torch.inf)
        #atten_scores.masked_fill(self.mask[:num_tokens, :num_tokens] == 1, float('-inf'))
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        atten_scores.masked_fill(mask_bool, -torch.inf) 
        atten_scores.masked_fill_(mask_bool, -torch.inf) 
        #  위 둘의 차이 있음. 

        ######################################
        ##### 예상 문제 4. 가중치를 스케일링하고, Softmax를 
        #######################################
        atten_weights = torch.softmax(atten_scores / keys.shape[-1]**0.5, dim=-1)

        ######################################
        ##### 예상 문제 5. dropout을 한다.
        #######################################
        atten_weights = self.drop_out(atten_weights)

        ######################################
        ##### 예상 문제 6. context_weight를 만든다. 
        #######################################
        context_vec = atten_weights @ values

        return context_vec

In [25]:
import torch

# 모델이 한번에 처리할 수 있는 Token 수
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

d_in = inputs.shape[1]   # 입력 차원 (d=3)  토큰의 vector 크기기
print("d_in: ", d_in)
d_out = 2               # Q, K, V의 출력 차원 (d=3)

batch = torch.stack((inputs, inputs), dim=0)
print("batch_size:", batch.shape[0])
# --- 실행 예시 ---
torch.manual_seed(123)

# 가정: batch 변수가 이미 정의되어 있다고 가정 (예: b=2, num_tokens=6, d_in=...)
# context_length는 모델이 허용하는 최대 길이이므로, 현재 배치의 길이와 같거나 더 길게 설정합니다.
context_length = batch.shape[1] 

print("context_length:", context_length)

ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

d_in:  3
batch_size: 2
context_length: 6
In init, context_length: 6
In forward, context_length: 6
In forward, num_tokens: 6
tensor([[[-0.5337, -0.1051],
         [-0.5323, -0.1080],
         [-0.5323, -0.1079],
         [-0.5297, -0.1076],
         [-0.5311, -0.1066],
         [-0.5299, -0.1081]],

        [[-0.5337, -0.1051],
         [-0.5323, -0.1080],
         [-0.5323, -0.1079],
         [-0.5297, -0.1076],
         [-0.5311, -0.1066],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])
